# 6. Fixed-depth versus column-vector PV representation

This notebook places the fixed-level estimates and the cached thickness-weighted 0–1000 m vector mean on one visual scorecard. Fixed levels use the same matched Eddy–Day sample.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "seacofs_tilt_tools.py").exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError("Run inside seacofs_eddy_tilt_analysis or one of its subfolders")
if str(ANALYSIS_ROOT) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_ROOT))
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import seacofs_tilt_tools as tilt
import depth_pv_tools as dpt

sns.set_theme(style="whitegrid", context="notebook")
DOMINANCE_FACTOR = 2.0
TARGET_DEPTHS_M = (0, 200, 500, 700, 1000)
MIN_TILT_KM = 5.0

depth_df = tilt.add_pv_gradient_terms(source="depth")
snapshot_df = tilt.add_pv_gradient_terms(source="depth_snapshot")
dpt.validate_depth_tables(depth_df, snapshot_df)
DEPTHS = dpt.nearest_cached_depths(depth_df, TARGET_DEPTHS_M)
comparison = dpt.add_surface_differences(depth_df, DOMINANCE_FACTOR)
matched = dpt.matched_depth_rows(comparison, DEPTHS)
palette = {"AE": "#c44e52", "CE": "#4c72b0"}
depth_cmap = plt.get_cmap("viridis")
depth_colours = dict(zip(DEPTHS, depth_cmap(np.linspace(.08, .92, len(DEPTHS)))))
depth_labels = {z: f"{z:g} m" for z in DEPTHS}


In [ ]:
use_depth = matched[matched.TiltDis.ge(MIN_TILT_KM)]
matched_keys = matched[["Eddy", "Day"]].drop_duplicates()
use_snapshot = snapshot_df.merge(matched_keys, on=["Eddy", "Day"], how="inner", validate="one_to_one")
use_snapshot = use_snapshot[use_snapshot.TiltDis.ge(MIN_TILT_KM)]
score = dpt.representation_scorecard(use_depth, use_snapshot, DEPTHS)
order = [f"{z:g} m" for z in DEPTHS] + ["0–1000 m vector mean"]
score["representation"] = pd.Categorical(score.representation, categories=order, ordered=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
metrics = [("median_error_deg", "Median error (degrees)", False),
           ("fraction_within_45", "Fraction within 45°", True),
           ("mean_alignment", "Mean polarity-aware cosine", True)]
for ax, (metric, label, higher) in zip(axes, metrics):
    sns.pointplot(data=score, x="representation", y=metric, hue="Cyc", palette=palette, dodge=.2, ax=ax)
    ax.tick_params(axis="x", rotation=35); ax.set(xlabel="", ylabel=label, title="Higher is better" if higher else "Lower is better")
    if ax is not axes[-1] and ax.legend_: ax.legend_.remove()
fig.suptitle("Which PV-gradient representation best describes tilt direction?")
plt.show()

In [ ]:
surface = dpt.surface_rows(depth_df)[["Eddy","Day","PV_grad_x","PV_grad_y"]].rename(columns={"PV_grad_x":"sx","PV_grad_y":"sy"})
column = snapshot_df[["Eddy","Day","PV_grad_x","PV_grad_y"]].merge(surface, on=["Eddy","Day"], validate="one_to_one")
column["rotation_deg"] = np.abs(dpt.angle_difference(
    np.degrees(np.arctan2(column.PV_grad_x, column.PV_grad_y)) % 360,
    np.degrees(np.arctan2(column.sx, column.sy)) % 360))
fig, ax = plt.subplots(figsize=(9,4.5), constrained_layout=True)
sns.histplot(column.rotation_deg, bins=np.arange(0,185,5), stat="density", element="step", fill=True, color="tab:purple", ax=ax)
ax.set(xlabel="Rotation between shallow and 0–1000 m vector-mean PV gradients (degrees)", title="The column mean is not necessarily the surface vector")
plt.show()

Select a preferred representation from consistent effect size, sample coverage and physical interpretation. The column-vector mean is a distinct whole-column hypothesis, not a substitute for examining where along the spine the signal changes.